# 第8章 木構造とアンサンブル学習

『Python機械学習スタートブック』のコードをGoogle Colabで実行するためのノートブックです。
コードは書籍のリスト番号順に並んでいます。上から順に実行してください。

- 解説（Web教材）: https://ml.kano.ac/chapters/8/
- 演習の解答: https://ml.kano.ac/solutions/8/

## 決定木

### scikit-learnによる決定木の実装

**リスト 8.1**　決定木によるIrisデータの分類

In [ ]:
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Irisデータセットの読み込み
df = sns.load_dataset("iris")
X = df.drop("species", axis=1)
y = df["species"]

# 訓練データとテストデータに分割
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# 決定木の学習
tree_clf = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_clf.fit(X_train, y_train)

# 予測と精度の評価
y_pred = tree_clf.predict(X_test)
print(f"正解率: {accuracy_score(y_test, y_pred):.4f}")

### 決定木の可視化

**リスト 8.2**　`plot_tree`による決定木の可視化

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(5.8, 4.3))
plot_tree(tree_clf,
          feature_names=X.columns.tolist(),
          class_names=tree_clf.classes_.tolist(),
          filled=True,
          rounded=True,
          fontsize=9,
          ax=ax)
plt.title("決定木（Iris, max_depth=3）")
plt.tight_layout()
plt.show()

### 過学習と決定木

**リスト 8.3**　`max_depth`を変えた精度の比較

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

for depth in [1, 2, 3, 5, 10, None]:
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42)
    clf.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, clf.predict(X_train))
    test_acc  = accuracy_score(y_test, clf.predict(X_test))
    print(f"max_depth={str(depth):>4s}  "
          f"訓練精度={train_acc:.4f}  テスト精度={test_acc:.4f}")

## ランダムフォレスト

### scikit-learnによるランダムフォレストの実装

**リスト 8.4**　決定木とランダムフォレストの比較

In [ ]:
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# ペンギンデータセットの読み込みと前処理
df = sns.load_dataset("penguins").dropna()
X = df[["bill_length_mm", "bill_depth_mm",
        "flipper_length_mm", "body_mass_g"]]
y = df["species"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# 単一の決定木
dt_clf = DecisionTreeClassifier(random_state=42)
dt_clf.fit(X_train, y_train)
dt_acc = accuracy_score(y_test, dt_clf.predict(X_test))

# ランダムフォレスト
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train, y_train)
rf_acc = accuracy_score(y_test, rf_clf.predict(X_test))

print(f"決定木（単体）の正解率:       {dt_acc:.4f}")
print(f"ランダムフォレストの正解率:    {rf_acc:.4f}")

### 決定木の本数と正解率の関係

**リスト 8.5**　決定木の本数と正解率の関係

In [ ]:
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

n_trees_list = [1, 5, 10, 20, 50, 100, 200, 500]
test_scores = []

for n_trees in n_trees_list:
    clf = RandomForestClassifier(n_estimators=n_trees, random_state=42)
    clf.fit(X_train, y_train)
    test_scores.append(accuracy_score(y_test, clf.predict(X_test)))

plt.figure(figsize=(5.8, 2.9))
plt.plot(n_trees_list, test_scores, marker="o")
plt.xlabel("決定木の本数")
plt.ylabel("テスト正解率")
plt.title("ランダムフォレスト：木の本数と正解率")
plt.xscale("log")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 勾配ブースティング

### scikit-learnによる勾配ブースティングの実装

**リスト 8.6**　勾配ブースティングによるペンギンの分類

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

# 勾配ブースティングの学習
gb_clf = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
gb_clf.fit(X_train, y_train)

y_pred = gb_clf.predict(X_test)
print(f"勾配ブースティングの正解率: {accuracy_score(y_test, y_pred):.4f}")

### 学習率の影響

**リスト 8.7**　学習率を変えた精度の比較

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

# ノイズを含むやや難しい人工データを生成
# （flip_y=0.1 でラベルの10%をランダムに反転）
X2, y2 = make_classification(
    n_samples=500, n_features=10, n_informative=5,
    n_redundant=2, flip_y=0.1, random_state=42
)
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.3, random_state=42
)

learning_rates = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
for lr in learning_rates:
    clf = GradientBoostingClassifier(
        n_estimators=100, learning_rate=lr,
        max_depth=3, random_state=42
    )
    clf.fit(X2_train, y2_train)
    train_acc = accuracy_score(y2_train, clf.predict(X2_train))
    test_acc = accuracy_score(y2_test, clf.predict(X2_test))
    print(f"learning_rate={lr}: "
          f"訓練 {train_acc:.4f}, テスト {test_acc:.4f}")

## 特徴量の重要度

### 各モデルの特徴量の重要度を比較

**リスト 8.8**　3つのモデルの特徴量の重要度の可視化

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 各モデルの学習（前節で学習済みのものを再利用）
models = {
    "Decision Tree":      dt_clf,
    "Random Forest":      rf_clf,
    "Gradient Boosting":  gb_clf,
}

feature_names = X.columns.tolist()

fig, axes = plt.subplots(1, 3, figsize=(5.8, 1.7))

for ax, (name, model) in zip(axes, models.items()):
    importances = model.feature_importances_
    indices = np.argsort(importances)
    ax.barh(range(len(indices)),
            importances[indices], align="center")
    ax.set_yticks(range(len(indices)))
    ax.set_yticklabels([feature_names[i] for i in indices])
    ax.set_xlabel("重要度")
    ax.set_title(name)

plt.tight_layout()
plt.show()

**リスト 8.9**　特徴量の重要度の数値の確認

In [ ]:
import pandas as pd

importance_df = pd.DataFrame({
    name: model.feature_importances_
    for name, model in models.items()
}, index=feature_names)

print(importance_df.round(4))

## 演習問題

### 演習 8-1: ジニ不純度の手計算

ジニ不純度について、次の場合の値を手計算で求めてください。

1. クラス A が 80%、クラス B が 20% のノード
2. クラス A が 50%、クラス B が 50% のノード
3. クラス A が 100% のノード（純粋なノード）

[解答例を見る](https://ml.kano.ac/solutions/8/#solution-8-1)

### 演習 8-2: 決定木による分類と過学習

penguins データセットを使って、`bill_length_mm`・`bill_depth_mm`・`body_mass_g` の 3 つの数値特徴量からペンギンの種類（`species`：Adelie / Chinstrap / Gentoo）を分類する決定木を実装してください。

**タスク**：

1. penguins データセットを読み込み、`dropna()` で欠損値のある行を削除する
2. 特徴量 `X = penguins[['bill_length_mm', 'bill_depth_mm', 'body_mass_g']]`、ラベル `y = penguins['species']` を準備する
3. データを「訓練 80%、テスト 20%」（`random_state=42`）に分割する
4. 決定木モデル `DecisionTreeClassifier(max_depth=3, random_state=42)` を学習する
5. テストデータで予測し、正解率（accuracy）と `classification_report` を表示する
6. `plot_tree` で学習した木構造を可視化する
7. `max_depth` を 1 から 10 まで変化させ、訓練精度とテスト精度をグラフにプロットする
8. 過学習が始まる `max_depth` はいくつか考察する

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# 1. データの読み込みと欠損値の削除

# 2. 特徴量 X とラベル y の設定

# 3. 訓練・テストに分割（8:2, random_state=42）

# 4. 決定木モデル（max_depth=3）の学習

# 5. テストデータで予測し、正解率と classification_report を表示

# 6. plot_tree で木構造を可視化

# 7. max_depth を 1〜10 で変化させ、訓練精度とテスト精度をプロット

[解答例を見る](https://ml.kano.ac/solutions/8/#solution-8-2)

### 演習 8-3: ランダムフォレストと特徴量の重要度

演習 8-2 と同じ penguins データセットの設定で、ペンギンの種類を分類するランダムフォレストを実装してください。また、特徴量の重要度を確認し、決定木との違いを考察してください。

**タスク**：

1. 演習 8-2 と同様に、データの読み込みから訓練・テスト分割までを行う
2. ランダムフォレスト `RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42)` を学習し、テストデータの正解率を表示する
3. `feature_importances_` を取り出し、特徴量名と一緒に [`pandas.DataFrame()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) にまとめて重要度の降順で表示する
4. 棒グラフで重要度を可視化する
5. `max_depth` を `1, 2, 3, 5, None` と変化させたときのテスト精度を比較する
6. 決定木（演習 8-2）と比べて何が変わったか考察する

In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. データの準備（演習 8-2 と同じ）

# 2. ランダムフォレストの学習と正解率の表示

# 3. 特徴量の重要度を DataFrame に整理して表示

# 4. 棒グラフで可視化

# 5. max_depth を変えたときのテスト精度を比較

[解答例を見る](https://ml.kano.ac/solutions/8/#solution-8-3)

### 演習 8-4: 学習率と木の本数の関係

演習 8-2 と同じ penguins データセットの設定で、勾配ブースティング（`GradientBoostingClassifier`）の `learning_rate` を 0.01 に固定し、`n_estimators` を 10, 50, 100, 500, 1000 と変化させたときのテスト精度を比較してください。学習率と決定木の本数の関係について考察してください。

[解答例を見る](https://ml.kano.ac/solutions/8/#solution-8-4)

### 発展課題: 特徴量重要度の比較

iris データセットに対して、決定木、ランダムフォレスト、勾配ブースティングの 3 つのモデルを学習し、それぞれの特徴量の重要度を比較する棒グラフを作成してください。演習 8-3 の penguins データセットの場合と比べて、どのような違いがあるか考察してください。

[解答例を見る](https://ml.kano.ac/solutions/8/#solution-8-adv-1)